# 🔗 Aula 24 — Conexão com o Projeto Integrador (PI)

**Curso:** Técnico em Inteligência Artificial | **UC:** Introdução a Arquitetura de Computadores

---

## 🎯 Situação de Aprendizagem

A UC chegou ao fim. Você aprendeu a montar servidores com GPU, escrever kernels CUDA, automatizar monitoramento e treinar modelos de IA reais. Agora o desafio é **não deixar esse conhecimento isolado**: o Projeto Integrador tem componentes de processamento de dados, modelos preditivos ou análise de imagens que podem ser acelerados com GPU. Nesta aula, o time mapeia essas conexões, integra a infraestrutura ao repositório do PI e documenta o plano de ação.

> ℹ️ O **PI do curso é uma pesquisa** e **não exige programação**. A aceleração GPU entra como **complemento opcional** que sustenta a recomendação.

Este notebook roda **sem GPU**.

In [ ]:
# 1. Preparar um mini-repositório de exemplo para a análise
import os

os.makedirs("pi_exemplo", exist_ok=True)

codigo_exemplo = '''
import numpy as np
import pickle

def processar_imagens(pasta):
    resultados = []
    for arquivo in os.listdir(pasta):   # loop sobre arquivos (CPU)
        img = carregar(arquivo)
        resultados.append(np.dot(img, pesos))   # np.dot pesado (CPU)
    return resultados

modelo = pickle.load(open("modelo.pkl", "rb"))   # carregamento de modelo
for epoch in range(10):                            # loop de treinamento
    previsoes = modelo.predict(dados)              # inferencia em loop
'''

with open("pi_exemplo/analise.py", "w", encoding="utf-8") as f:
    f.write(codigo_exemplo)

print("Repositório de exemplo criado em 'pi_exemplo/'.")
print(open("pi_exemplo/analise.py", encoding="utf-8").read())

## 1. Mapear conexões GPU ↔ PI automaticamente

Varredura do repositório à procura de padrões de alto custo computacional.

In [ ]:
# 2. Detectar candidatos a aceleração por GPU
import re
from pathlib import Path

PADROES_GPU = {
    "Loop sobre imagens/arquivos": re.compile(r"for .* in .*(imagens?|arquivos?|images?|files?)", re.I),
    "Carregamento de modelo ML":   re.compile(r"(load_model|from_pretrained|pickle.load|joblib.load)", re.I),
    "Operacoes numpy intensivas":  re.compile(r"np\.(dot|matmul|linalg|fft|einsum)", re.I),
    "Loop de treinamento":         re.compile(r"for .*(epoch|batch|step)", re.I),
    "Inferencia em loop":          re.compile(r"(model\.predict|\.forward|sess\.run)", re.I),
}

def analisar_repositorio(raiz="."):
    resultados = {k: [] for k in PADROES_GPU}
    for caminho in Path(raiz).rglob("*.py"):
        if any(p in str(caminho) for p in [".venv", "node_modules", "__pycache__"]):
            continue
        texto = caminho.read_text(errors="ignore")
        for desc, regex in PADROES_GPU.items():
            m = regex.findall(texto)
            if m:
                resultados[desc].append((str(caminho), len(m)))
    return resultados

resultados = analisar_repositorio("pi_exemplo")
total = 0
for desc, ocorrencias in resultados.items():
    if ocorrencias:
        for arq, n in ocorrencias:
            print(f"🔍 {desc}: {arq} ({n} ocorrencia(s))")
            total += n
print(f"\n{total} ocorrencia(s) com potencial de aceleracao GPU.")

## 2. Plano de ação por domínio

Cada domínio tem acelerações e configurações de monitor específicas.

In [ ]:
# 3. Definir os planos por domínio
PLANOS = {
    "visao": {
        "titulo": "Visao Computacional no PI",
        "aceleracoes": [
            ("DataLoader multi-worker", "num_workers=4, pin_memory=True"),
            ("Mixed Precision (AMP)", "torch.amp.autocast + GradScaler"),
            ("Batch maior", "RTX 4090: 64-128 imagens"),
            ("Transfer Learning", "ResNet/ViT pre-treinado + fine-tuning"),
        ],
        "monitor": {"intervalo": 10, "max_temp": 83, "max_vram": 88},
    },
    "nlp": {
        "titulo": "NLP / Texto no PI",
        "aceleracoes": [
            ("Gradient checkpointing", "economiza 30-40% de VRAM"),
            ("Gradient accumulation", "batch grande com VRAM de batch 32"),
            ("Tokenizacao paralela", "tokenizer(batch, num_proc=4)"),
            ("FP16 inference", "model.half() em producao"),
        ],
        "monitor": {"intervalo": 15, "max_temp": 80, "max_vram": 92},
    },
    "series_temporais": {
        "titulo": "Series Temporais no PI",
        "aceleracoes": [
            ("LSTM/Transformer na GPU", "sequencias em .cuda()"),
            ("Janela vetorizada", "torch.unfold, sem loop Python"),
            ("Normalizacao na GPU", "mean/std no tensor CUDA"),
            ("Workers persistentes", "persistent_workers=True"),
        ],
        "monitor": {"intervalo": 20, "max_temp": 82, "max_vram": 85},
    },
}

def gerar_plano(dominio):
    plano = PLANOS[dominio]
    print(f"=== {plano['titulo']} ===\n")
    for i, (tec, desc) in enumerate(plano["aceleracoes"], 1):
        print(f"  {i}. {tec}\n     -> {desc}")
    cfg = plano["monitor"]
    print(f"\n  Monitor: {cfg['intervalo']}s | {cfg['max_temp']}C | {cfg['max_vram']}% VRAM\n")

gerar_plano("visao")

## 3. Estimar o ganho de velocidade no PI

Relação entre o tempo de uma tarefa em CPU e o tempo estimado em GPU.

In [ ]:
# 4. Estimar o speedup de um componente do PI
def estimar_ganho(tempo_cpu_s, speedup_gpu, tempo_transferencia_s=0.0):
    """Tempo em GPU = tempo CPU / speedup + transferencia (overhead unico)."""
    tempo_gpu = tempo_cpu_s / speedup_gpu + tempo_transferencia_s
    return tempo_gpu, tempo_cpu_s / tempo_gpu

componentes = [
    ("Pre-processamento de imagens", 3600, 8, 5),
    ("Inferencia de 10k amostras",    1800, 12, 3),
    ("Treino do modelo",              7200, 10, 10),
]

print(f"{'Componente':<30} {'CPU (s)':>9} {'GPU (s)':>9} {'Ganho':>8}")
print("-" * 60)
ganho_total_cpu = 0
ganho_total_gpu = 0
for nome, cpu_s, sp, trans in componentes:
    gpu_s, ganho = estimar_ganho(cpu_s, sp, trans)
    ganho_total_cpu += cpu_s
    ganho_total_gpu += gpu_s
    print(f"{nome:<30} {cpu_s:>9} {gpu_s:>9.0f} {ganho:>7.1f}x")
print("-" * 60)
print(f"TOTAL: CPU {ganho_total_cpu}s -> GPU {ganho_total_gpu:.0f}s "
      f"({ganho_total_cpu/ganho_total_gpu:.1f}x mais rapido)")

## 4. Integrar a infraestrutura de monitoramento

Simulação (sem copiar arquivos de verdade) de como a infraestrutura GPU entra no PI.

In [ ]:
# 5. Gerar o esqueleto de integração da infraestrutura GPU no PI
ESTRUTURA_PI = [
    "scripts/gpu/monitor_treinamento.sh",
    "scripts/gpu/alertas.sh",
    "scripts/gpu/dashboard_metricas.py",
    "scripts/gpu/relatorio_final.py",
    "logs/monitor/",
    "logs/dashboards/",
]

print("Estrutura a adicionar ao repositorio do PI:")
for item in ESTRUTURA_PI:
    print(f"  + {item}")

print("\nVariaveis de ambiente necessarias (.env.example):")
for var in ["SLACK_WEBHOOK_URL", "TELEGRAM_BOT_TOKEN", "TELEGRAM_CHAT_ID", "ALERTA_EMAIL", "WANDB_API_KEY"]:
    print(f"  {var}=")

---
## 📝 5. Exercícios Práticos (5)

Resolva os 5 exercícios a seguir completando as células de código correspondentes.

### Exercício 1: Mapear o seu próprio repositório

Aponte `analisar_repositorio` para a pasta do seu PI (ou `.`) e liste os candidatos encontrados.

In [ ]:
# Escreva a sua solução para o Exercício 1 abaixo:
# res = analisar_repositorio('.')

### Exercício 2: Gerar o plano do seu domínio

Chame `gerar_plano` para o domínio do seu PI (visao, nlp ou series_temporais).

In [ ]:
# Escreva a sua solução para o Exercício 2 abaixo:
# gerar_plano('nlp')

### Exercício 3: Estimar o ganho de um componente real

Use `estimar_ganho` com o tempo em CPU de um componente do seu PI e um speedup plausível (8×). Imprima o resultado.

In [ ]:
# Escreva a sua solução para o Exercício 3 abaixo:
# print(estimar_ganho(1200, 8, 4))

### Exercício 4: Definir os thresholds do monitor do PI

Crie um dicionário com `intervalo`, `max_temp` e `max_vram` adequados ao seu domínio e explique a escolha.

In [ ]:
# Escreva a sua solução para o Exercício 4 abaixo:
# meu_monitor = {'intervalo': 15, 'max_temp': 80, 'max_vram': 90}

### Exercício 5: As 3 conexões GPU ↔ PI

Escreva e imprima as **3 conexões** mais relevantes entre o projeto de GPU e o seu PI, no formato `conexão -> ganho esperado`.

In [ ]:
# Escreva a sua solução para o Exercício 5 abaixo:
# conexoes = [("...", "..."), ("...", "..."), ("...", "...")]

---
## 📌 6. Síntese Final da UC

**Jornada completa:**
- **Bloco 1 — Fundamentos:** arquitetura CPU vs GPU, CUDA, VRAM, benchmark e primeiros scripts Bash.
- **Bloco 2 — Programação:** CUDA, OpenCL, ROCm, kernels, profiling e otimização de memória.
- **Bloco 3 — Automação:** monitoramento, filas de jobs e gestão de energia.
- **Bloco 4 — Projeto Final:** pipeline completo + W&B + monitor + dashboard + pitch + conexão com o PI.

**Entregas finais da UC:** repositório do projeto GPU (README + requirements), dashboard PNG,
monitor integrado com log CSV, documento "3 conexões GPU ↔ PI" e avaliação formativa.